In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("catalogo", "bank_dev")
dbutils.widgets.text("esquema_silver", "silver")
dbutils.widgets.text("esquema_gold", "gold")
dbutils.widgets.text("storageName", "saprojectbankdeveastus")
dbutils.widgets.text("containerSilver", "silver")
dbutils.widgets.text("containerGold", "gold")

In [0]:
catalogo         = dbutils.widgets.get("catalogo")
esquema_silver   = dbutils.widgets.get("esquema_silver")
esquema_gold     = dbutils.widgets.get("esquema_gold")
storageName      = dbutils.widgets.get("storageName")
containerSilver  = dbutils.widgets.get("containerSilver")
containerGold    = dbutils.widgets.get("containerGold") 


In [0]:
# Nombres de las tablas Silver (Origen)
tabla_silver_tx = f"{catalogo}.{esquema_silver}.transactions" 
tabla_silver_users = f"{catalogo}.{esquema_silver}.dim_users"
tabla_silver_cards = f"{catalogo}.{esquema_silver}.dim_cards"
tabla_silver_mcc = f"{catalogo}.{esquema_silver}.dim_mcc_codes"
tabla_silver_fraud = f"{catalogo}.{esquema_silver}.dim_fraud_labels"


In [0]:
# Nombre de la tabla Gold (Destino)
tabla_gold_datamart = f"{catalogo}.{esquema_gold}.fraud_transaction_datamart"

In [0]:
from pyspark.sql import functions as F



### Leyendo tablas de la capa Silver

In [0]:
# 1. Tabla de Hechos (Transacciones) con streaming
df_tx = spark.read.table(tabla_silver_tx)

# 2. Dimensiones con Historial (SCD2) con streaming
df_users = spark.read.table(tabla_silver_users)
df_cards = spark.read.table(tabla_silver_cards)

# 3. Dimensiones Estáticas (Catálogos)
df_mcc = spark.read.table(tabla_silver_mcc)
df_fraud = spark.read.table(tabla_silver_fraud)


### Realizando cruces Point-in-Time...

In [0]:
# ===================== POINT-IN-TIME JOIN & ENRIQUECIMIENTO =====================

df_users_stg = df_users.select(
    F.col("id").alias("user_dim_id"),
    F.col("birth_date"),
    F.col("gender").alias("client_gender"),
    F.col("address"),
    "latitude", "longitude",
    "per_capita_income", "yearly_income", "total_debt",
    "credit_score", "num_credit_cards",
    "valid_from", "valid_to", "is_current"
)

df_cards_stg = df_cards.select(
    F.col("id").alias("card_dim_id"),  
    "card_brand", "card_type", "card_on_dark_web",
    "valid_from", "valid_to", "is_current"
)

df_mcc_stg = df_mcc.select(
    F.col("mcc_code").alias("mcc_code"),
    F.col("mcc_description").alias("mcc_description")
)

# 1. Join con Users (Point-in-Time adaptado para datos históricos)
df_gold_step1 = df_tx.join(
    df_users_stg,
    (df_tx.client_id == df_users_stg.user_dim_id) & 
    (
        # Condición normal de SCD2
        (df_tx.date >= df_users_stg.valid_from) | 
        # Excepción: Si es el registro inicial/actual y la transacción es muy vieja
        (df_users_stg.is_current == True) 
    ) & 
    (df_tx.date <= df_users_stg.valid_to),
    "left"
)

# 2. Join con Cards (Point-in-Time)
df_gold_step2 = df_gold_step1.join(
    df_cards_stg,
    (df_gold_step1.card_id == df_cards_stg.card_dim_id) &
    (df_gold_step1.date >= df_cards_stg.valid_from) &
    (df_gold_step1.date <= df_cards_stg.valid_to),
    "left"
)

# 3. Join con MCC (simple)
df_gold_step3 = df_gold_step2.join(
    df_mcc_stg,
    df_gold_step2.mcc == df_mcc_stg.mcc_code,
    "left"
)

# 5. Cruce con Fraud Labels
df_gold_final = df_gold_step3.join(
    df_fraud,
    df_gold_step3.id == df_fraud.transaction_id,
    "left"
).drop(df_fraud.silver_load_date, df_fraud.transaction_id)


### Enriquecimiento para negocio

In [0]:
# ===================== ENRIQUECIMIENTO DE NEGOCIO (BUSINESS FEATURES) =====================
df_gold_final = df_gold_final \
    .withColumn("is_fraud", F.coalesce(F.col("is_fraud"), F.lit(False))) \
    .drop(
        "user_dim_id", "card_dim_id", "mcc_code", 
        "valid_from", "valid_to", "is_current"   # eliminamos columnas de control SCD
    )



df_gold_business = df_gold_final \
    .withColumn("transaction_year", F.year(F.col("date"))) \
    .withColumn("transaction_month", F.month(F.col("date"))) \
    .withColumn("transaction_hour", F.hour(F.col("date"))) \
    .withColumn("day_of_week", F.dayofweek(F.col("date"))) \
    .withColumn("is_weekend", F.when(F.col("day_of_week").isin(1, 7), True).otherwise(False)) \
    .withColumn("customer_age", F.floor(F.datediff(F.col("date"), F.col("birth_date")) / 365.25)) \
    .withColumn(
        "age_group", 
        F.when(F.col("customer_age") < 25, "1. Menos de 25")
         .when((F.col("customer_age") >= 25) & (F.col("customer_age") <= 35), "2. 25 a 35")
         .when((F.col("customer_age") >= 36) & (F.col("customer_age") <= 50), "3. 36 a 50")
         .otherwise("4. Mayores de 50")
    ) \
    .withColumn("debt_to_income_ratio", F.round(F.col("total_debt") / F.col("yearly_income"), 4)) \
    .withColumn(
        "transaction_channel", 
        F.when(F.col("use_chip").contains("Online"), "Online")
         .when(F.col("use_chip").contains("Swipe"), "Banda Magnética")
         .when(F.col("use_chip").contains("Chip"), "Chip Físico")
         .otherwise("Otro")
    )

###  Eliminamos columnas no importantes y redundantes)

In [0]:
# ===================== LIMPIEZA DE COLUMNAS TÉCNICAS =====================
columnas_a_eliminar = [
    "user_dim_id", "card_dim_id", "mcc_code", 
    "valid_from", "valid_to", "is_current", "silver_load_date", 
    "birth_date", "day_of_week" 
]

# Solo borramos las columnas si existen en el DataFrame para evitar errores
columnas_existentes = df_gold_business.columns
columnas_finales_borrar = [col for col in columnas_a_eliminar if col in columnas_existentes]

df_gold_ready_for_bi = df_gold_business.drop(*columnas_finales_borrar)


### Guardamos la tabla limpia y lista

In [0]:
# ===================== PARTE 3: GUARDADO EN GOLD =====================
(df_gold_ready_for_bi.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true") # Permite actualizar el esquema de la tabla si agregamos columnas nuevas
 .saveAsTable(tabla_gold_datamart))
